In [ ]:
# ================================================
# 0 — IMPORTS
# ================================================
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GroupKFold
from catboost import CatBoostClassifier, Pool
import optuna

# ================================================
# 1 — LOAD
# ================================================
train = pd.read_csv("datasets/training_data.csv", encoding="latin1")
test = pd.read_csv("datasets/test_data.csv", encoding="latin1")

y = train["AVERAGE_SPEED_DIFF"].fillna("None").astype(str)
X = train.drop("AVERAGE_SPEED_DIFF", axis=1)

# ================================================
# 2 — DATE FEATURES
# ================================================
def process_dates(df):
    df = df.copy()
    df["record_date"] = pd.to_datetime(df["record_date"])

    df["day"] = df["record_date"].dt.day
    df["month"] = df["record_date"].dt.month
    df["hour"] = df["record_date"].dt.hour
    df["weekday"] = df["record_date"].dt.weekday
    df["date_group"] = df["record_date"].dt.date.astype(str)   # <- IMPORTANT !!

    df = df.drop(columns=["record_date"])
    return df

X = process_dates(X)
test = process_dates(test)

# Guardar grupos (dias)
groups = X["date_group"]
X = X.drop(columns=["date_group"])  # removido do treino
test = test.drop(columns=["date_group"])

# ================================================
# 3 — FIX NUMERIC
# ================================================
numeric_cols = [
    "AVERAGE_FREE_FLOW_SPEED",
    "AVERAGE_TIME_DIFF",
    "AVERAGE_FREE_FLOW_TIME",
    "AVERAGE_TEMPERATURE",
    "AVERAGE_ATMOSP_PRESSURE",
    "AVERAGE_HUMIDITY",
    "AVERAGE_WIND_SPEED",
    "AVERAGE_PRECIPITATION"
]

for c in numeric_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce").fillna(0)
    test[c] = pd.to_numeric(test[c], errors="coerce").fillna(0)

# ================================================
# 4 — LIGHT FEATURE ENGINEERING (dataset pequeno)
# ================================================
def add_features(df):
    df = df.copy()

    df["is_weekend"] = (df["weekday"] >= 5).astype(int)

    df["rush_hour"] = (
        df["hour"].between(7, 9) |
        df["hour"].between(17, 19)
    ).astype(int)

    df["time_ratio"] = (
        df["AVERAGE_TIME_DIFF"] + 1e-6
    ) / (
        df["AVERAGE_FREE_FLOW_TIME"] + 1e-6
    )

    df["weather_index"] = (
        df["AVERAGE_PRECIPITATION"] *
        df["AVERAGE_HUMIDITY"]
    )

    return df

X = add_features(X)
test = add_features(test)

# ================================================
# 5 — CLEAN CATEGORICALS
# ================================================
def clean_cats(df):
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == object:
            df[col] = df[col].astype(str).str.strip()
            df[col] = df[col].replace("", "Unknown").fillna("Unknown")
    return df

X = clean_cats(X)
test = clean_cats(test)

cat_cols = [c for c in X.columns if X[c].dtype == object]
cat_idx = [X.columns.get_loc(c) for c in cat_cols]

# ================================================
# 6 — GROUPKFold por DIA! (REAL Kaggle-like)
# ================================================
gkf = GroupKFold(n_splits=5)

train_idx, val_idx = next(gkf.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

train_pool = Pool(X_train, y_train, cat_features=cat_idx)
val_pool = Pool(X_val, y_val, cat_features=cat_idx)

# ================================================
# 7 — OPTUNA (forte, mas seguro)
# ================================================
def objective(trial):

    params = {
        "iterations": trial.suggest_int("iterations", 900, 1500),
        "depth": trial.suggest_int("depth", 8, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.06),

        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 5, 25),

        "bootstrap_type": "Bayesian",
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.3, 2.0),

        "loss_function": "MultiClass",
        "eval_metric": "Accuracy",
        "random_seed": 42,
        "od_type": "Iter",
        "od_wait": 80,
        "verbose": False
    }

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=val_pool, verbose=False, use_best_model=True)

    preds = model.predict(X_val).reshape(-1)
    return accuracy_score(y_val, preds)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=40)

best_params = study.best_params
print("\nBEST PARAMS:", best_params)

# ================================================
# 8 — TRAIN FINAL MODEL
# ================================================
best = CatBoostClassifier(
    **best_params,
    loss_function="MultiClass",
    eval_metric="Accuracy",
    random_seed=42,
    od_type="Iter",
    od_wait=80,
    verbose=100
)

full_pool = Pool(X, y, cat_features=cat_idx)
best.fit(full_pool)

# ================================================
# 9 — FINAL ACC (VALIDAÇÃO REALISTA!)
# ================================================
val_pred = best.predict(X_val).reshape(-1)
acc = accuracy_score(y_val, val_pred)

print("\n==============================")
print(f"ACC FINAL VALIDATION : {acc:.6f}")
print("==============================\n")

print(classification_report(y_val, val_pred))

# ================================================
# 10 — SUBMISSION (1..1500)
# ================================================
test_pool = Pool(test, cat_features=cat_idx)
test_pred = best.predict(test_pool).reshape(-1)

submission = pd.DataFrame({
    "RowId": np.arange(1, len(test_pred) + 1),
    "Speed_Diff": test_pred
})

submission.to_csv("submission.csv", index=False)
print("✔ submission.csv criado com sucesso!")


[I 2025-11-28 10:44:04,690] A new study created in memory with name: no-name-269e7872-735e-4813-a55e-d899f6a3018e


359
['2019-08-29' '2018-08-10' '2019-09-01' '2019-02-26' '2019-06-06'
 '2018-11-15' '2018-10-03' '2018-08-25' '2019-06-30' '2019-04-20'
 '2019-08-25' '2019-07-18' '2019-01-20' '2019-01-25' '2019-09-27'
 '2019-02-08' '2019-09-08' '2018-12-09' '2018-12-02' '2019-02-17']


[I 2025-11-28 10:44:14,208] Trial 0 finished with value: 0.8013196480938416 and parameters: {'iterations': 531, 'depth': 7, 'learning_rate': 0.07366357287751774, 'l2_leaf_reg': 8.446831694632918, 'bagging_temperature': 0.6291321528745378}. Best is trial 0 with value: 0.8013196480938416.
[I 2025-11-28 10:44:24,476] Trial 1 finished with value: 0.7917888563049853 and parameters: {'iterations': 522, 'depth': 7, 'learning_rate': 0.03648612305045173, 'l2_leaf_reg': 14.15760139285501, 'bagging_temperature': 1.249222222662468}. Best is trial 0 with value: 0.8013196480938416.
[I 2025-11-28 10:44:29,637] Trial 2 finished with value: 0.7925219941348973 and parameters: {'iterations': 454, 'depth': 6, 'learning_rate': 0.08335515651555267, 'l2_leaf_reg': 13.754150781181306, 'bagging_temperature': 2.8894757331731045}. Best is trial 0 with value: 0.8013196480938416.
[I 2025-11-28 10:45:01,569] Trial 3 finished with value: 0.7859237536656891 and parameters: {'iterations': 644, 'depth': 10, 'learning_r